# Project Planning Stage (Individual): Tilly Bauer

In [1]:
### Run this cell before continuing.
import altair as alt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.compose import make_column_transformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

## 1) Data Description

In [2]:
url = "https://drive.google.com/uc?export=download&id=1Mw9vW0hjTJwRWx0bDXiSpYsO3gKogaPz"
players_data = pd.read_csv(url)
players_data

,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
0,Pro,True,f6daba428a5e19a3d47574858c13550499be23603422e6...,30.3,Morgan,Male,9,NaN,NaN
1,Veteran,True,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,3.8,Christian,Male,17,NaN,NaN
2,Veteran,False,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,0.0,Blake,Male,17,NaN,NaN
3,Amateur,True,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4f...,0.7,Flora,Female,21,NaN,NaN
4,Regular,True,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb...,0.1,Kylie,Male,21,NaN,NaN
...,...,...,...,...,...,...,...,...,...
191,Amateur,True,b6e9e593b9ec51c5e335457341c324c34a2239531e1890...,0.0,Bailey,Female,17,NaN,NaN
192,Veteran,False,71453e425f07d10da4fa2b349c83e73ccdf0fb3312f778...,0.3,Pascal,Male,22,NaN,NaN
193,Amateur,False,d572f391d452b76ea2d7e5e53a3d38bfd7499c7399db29...,0.0,Dylan,Prefer not to say,17,NaN,NaN
194,Amateur,False,f19e136ddde68f365afc860c725ccff54307dedd13968e...,2.3,Harlow,Male,17,NaN,NaN


In [3]:
players_data_missing = players_data.isna().sum()
players_data_missing # shows that all rows of "individualId" and "organizationName" columns are missing

experience            0
subscribe             0
hashedEmail           0
played_hours          0
name                  0
gender                0
age                   0
individualId        196
organizationName    196
dtype: int64

The players data shown above contains the following 197 observations (rows) and 9 variables (columns), including:
1. experience: Categorical variable containing the player’s experience level.
2. subscribe: Categorical variable with whether player is subscribed or not subscribed to a game-related newsletter.
3. hashedEmail: Nominal variable that uniquely identifies each player.
4. played_hours: Numeric variable with total hours spent on Mine Craft server.
5. name: Nominal variable with the player’s name.
6. gender: Nominal variable with the player’s gender.
7. age: Discrete numeric variable with the player’s age.
8. individualId: NO DATA.
9. organizationName: NO DATA.

Issue:
- As shown in the players_data_missing, all row of the columns "individualId" and "organizationName" are missing data.

## (2) Question

I propose that our group should address the question: "How do the number of hours played and a player's experience level influence their likelihood of subscribing to the game newsletter?"

The players dataset contains all the necessary variables for this analysis. The explanatory variables are “played_hours” (total hours played by each player) and “experience” (player's level of experience). The response variable is “subscribe” (whether the player subscribed). By examining the relationship between the number of hours played and experience level, we can investigate whether players who are more engaged and have greater experience are more likely to subscribe.

To prepare the data for analysis, I will select only the relevant columns: “played_hours,” “experience,” and “subscribe.” Since the predictive methods we learned rely on numerical inputs, I will convert the categorical “experience” variable into an ordinal numerical format, assigning increasing values from 1 (beginner) to 5 (pro) according to experience level. Both predictors will be standardized to ensure equal contribution to the analysis. Before modeling, I will split the data into an 80/20 training and test set, allowing us to evaluate model performance on unseen data.

## (3) Exploratory Data Analysis and Visualization

In [4]:
# Tidy the data by isolating three columns required for model
columns_to_drop = ['hashedEmail', 'name','individualId', 'organizationName', 'gender', 'age']
players_data_tidy = players_data.drop(columns_to_drop, axis=1)

# Re-label subscribe "True" as "subscribed", and subscribe "False" as "not subscribed"
players_data_tidy['subscribe'] = players_data_tidy['subscribe'].replace({
    True : "subscribed",
    False : "not subscribed"
})
# Re-label experience levels with numbers
players_data_tidy['experience'] = players_data_tidy['experience'].replace({
    'Beginner' : 1,
    'Amateur' : 2,
    'Regular' : 3,
    'Veteran' : 4,
    'Pro' : 5
})

players_data_tidy

/tmp/ipykernel_1233/969487315.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  players_data_tidy['experience'] = players_data_tidy['experience'].replace({


,experience,subscribe,played_hours
0,5,subscribed,30.3
1,4,subscribed,3.8
2,4,not subscribed,0.0
3,2,subscribed,0.7
4,3,subscribed,0.1
...,...,...,...
191,2,subscribed,0.0
192,4,not subscribed,0.3
193,2,not subscribed,0.0
194,2,not subscribed,2.3


In [5]:
# Count the number of occurrences of each value in subscribe.
players_data_tidy["subscribe"].value_counts()

subscribe
subscribed        144
not subscribed     52
Name: count, dtype: int64

In [6]:
players_plot = alt.Chart(players_data_tidy).mark_bar().encode(
    x=alt.X("played_hours").title("Play time in hours"),
    y=alt.Y("count()").title("Number of players"),
    color = alt.Color("subscribe").title("Subscription status"),
).facet(
    "subscribe:N",
    columns = 2,
    title = "Number of players subscribed based on play time in hours"
)

players_plot

alt.FacetChart(...)

In [7]:
subscription_proportion = players_data.groupby('experience')['subscribe'].mean()
subscription_proportion_df = pd.DataFrame(subscription_proportion).reset_index()
experience_percent_plot = alt.Chart(subscription_proportion_df, title=["Percentage of players subscribed", "from each expereince level"]).mark_bar().encode(
    x=alt.X("experience").title("Player experience level"),
    y=alt.Y('subscribe').title("Percentage of players subscribed"),
)
experience_percent_plot

alt.Chart(...)

In [8]:
predictors_plot = alt.Chart(players_data, title="Relationship between experience level and hours played").mark_bar().encode(
    x=alt.X("played_hours").title("Total play time in hours"),
    y=alt.Y("experience").title("Player experience level"),
    color=alt.Color("experience").title("Player experience level")
)
predictors_plot

alt.Chart(...)

I first explored the relationship between each predictor and newsletter subscription. The histogram of total hours played, faceted by subscription status, shows that beyond a certain threshold of hours played, all players are subscribed to the newsletter. This pattern suggests that higher engagement, as measured by playtime, is associated with the likelihood of subscription. The bar chart of subscription rates by experience level indicates that “regulars” have the highest subscription rate, while other groups, including veterans, have lower rates. This pattern suggests experience level is an important, but not strictly linear, predictor.

The final visualization (a bar graph of play time by experience group) investigates the relationship between the two predictors. The plot indicates that amateurs and regulars have the most hours played, while other groups contribute less overall playtime. Despite some relationship between variables, there remains meaningful variation in hours played within each experience group. This suggests that both predictors provide distinct information for modeling subscription status, reducing concerns about redundancy and supporting the inclusion of both features in our analysis.

In [9]:
# Split stratify data and to ensure that the training and testing subsets 
# contain the right proportions of each category of observation.
players_train, players_test = train_test_split(
    players_data_tidy, train_size=0.8, stratify=players_data_tidy["subscribe"]
)
players_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 156 entries, 51 to 20
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   experience    156 non-null    int64  
 1   subscribe     156 non-null    object 
 2   played_hours  156 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 4.9+ KB


In [10]:
players_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40 entries, 100 to 162
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   experience    40 non-null     int64  
 1   subscribe     40 non-null     object 
 2   played_hours  40 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.2+ KB


We can see from the info method above that the training set contains 156 observations, while the test set contains 40 observations. This corresponds to the desired train/test split of 80/20.

In [11]:
# Create a pipline for KNN classification
players_preprocessor = make_column_transformer(
    (StandardScaler(), ["experience", "played_hours"]),
)
knn = KNeighborsClassifier(n_neighbors=3)

X = players_train[["experience", "played_hours"]]
y = players_train["subscribe"]

knn_pipeline = make_pipeline(players_preprocessor, knn)
knn_pipeline.fit(X, y)

knn_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('standardscaler',
                                                  StandardScaler(),
                                                  ['experience',
                                                   'played_hours'])])),
                ('kneighborsclassifier', KNeighborsClassifier(n_neighbors=3))])

With the pipeline complete, all necessary wrangling has been performed. 

## (4) Methods and Plan

I propose using K-nearest neighbors (KNN) classification to address our question. KNN is well-suited for binary classification tasks like predicting newsletter subscription (subscribe or not subscribe).

KNN classification does not require assumptions about the underlying distribution or linearity of the data, which is advantageous here since we do not know the precise relationship between predictors and outcome. The primary assumption underlying KNN classification is that individuals with similar predictor values—here, experience and hours played—will have similar outcomes regarding newsletter subscription. 

However, KNN has several limitations. It can become computationally slow with large datasets or a large number of predictors. Since the classes are slightly imbalanced (144 subscribers and 52 non-subscribers), the model may favor the majority class, especially if the K value is small. Additionally, standardizing predictors is essential, as KNN is distance-based and sensitive to scale differences.

To select and compare models, we will tune the number of neighbors (K) by testing a range of values and evaluating their performance. We will start by splitting the data into training (80%) and test (20%) sets. Within the training set, we will use 5-fold cross-validation to further divide the data into five equal parts, training on four and validating on one in each fold. This process provides an estimate of model accuracy for each K value. After identifying the optimal K, we will train the final KNN classifier on the entire training set and evaluate its performance on the test set to assess its generalizability.